In [123]:
import pandas as pd

# Simulating dataset loading (assuming CSV-like input)
data = pd.read_csv('retail_store_sales.csv')  # Replace with actual file path if needed

In [124]:
for col in ['Price Per Unit', 'Quantity', 'Total Spent']:
    data[col] = pd.to_numeric(data[col], errors='coerce')

In [125]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [126]:
data.describe()

,Price Per Unit,Quantity,Total Spent
count,11966.000000,11971.000000,11971.000000
mean,23.365912,5.536380,129.652577
std,10.743519,2.857883,94.750697
min,5.000000,1.000000,5.000000
25%,14.000000,3.000000,51.000000
50%,23.000000,6.000000,108.500000
75%,33.500000,8.000000,192.000000
max,41.000000,10.000000,410.000000


In [127]:
data.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [128]:
data.isnull().sum()

Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

In [129]:
# Step 1: Remove rows with two or more missing numerical values (Total Spent, Price Per Unit, Quantity)
missing_combinations = data[
    (data[['Total Spent', 'Price Per Unit', 'Quantity']].isna().sum(axis=1) >= 2)
]
print(f"\nRemoving {len(missing_combinations)} rows with 2 or more missing numerical values (Total Spent, Price Per Unit, Quantity):")
print(missing_combinations[['Transaction ID', 'Price Per Unit', 'Quantity', 'Total Spent']])

# Drop these rows
data = data[~(data[['Total Spent', 'Price Per Unit', 'Quantity']].isna().sum(axis=1) >= 2)]

# Now handle single missing values with calculations
# Fill Total Spent
mask_ts = data['Total Spent'].isna()
data.loc[
    mask_ts & data['Quantity'].notna() & data['Price Per Unit'].notna(),
    'Total Spent'
] = data['Quantity'] * data['Price Per Unit']

# Fill Quantity
mask_q = data['Quantity'].isna()
data.loc[
    mask_q & data['Total Spent'].notna() & data['Price Per Unit'].notna() & (data['Price Per Unit'] != 0),
    'Quantity'
] = (data['Total Spent'] / data['Price Per Unit']).round()

# Fill Price Per Unit
mask_ppu = data['Price Per Unit'].isna()
data.loc[
    mask_ppu & data['Total Spent'].notna() & data['Quantity'].notna() & (data['Quantity'] != 0),
    'Price Per Unit'
] = data['Total Spent'] / data['Quantity']

# Step 2: Handle remaining missing numerical values
for col in ['Price Per Unit', 'Quantity', 'Total Spent']:
    data[col] = data.groupby('Category')[col].transform(
        lambda x: x.fillna(x.median() if not pd.isna(x.median()) else data[col].median())
    )

# Step 3: Handle missing Item values
for idx, row in data[data['Item'].isna() & data['Price Per Unit'].notna() & data['Total Spent'].notna()].iterrows():
    expected_quantity = row['Total Spent'] / row['Price Per Unit']
    if expected_quantity.is_integer() and not pd.isna(expected_quantity):
        data.at[idx, 'Quantity'] = expected_quantity
        data.at[idx, 'Item'] = f"Unknown_{row['Category']}"
    else:
        data.at[idx, 'Item'] = f"Unknown_{row['Category']}"

# Fill remaining missing Item
data['Item'] = data['Item'].fillna('Unknown_Item')

# Step 4: Handle other categorical columns
data['Category'] = data['Category'].fillna(
    data['Category'].mode()[0] if not data['Category'].mode().empty else 'Unknown'
)

data['Payment Method'] = data['Payment Method'].fillna(
    data['Payment Method'].mode()[0] if not data['Payment Method'].mode().empty else 'Unknown'
)

data['Location'] = data['Location'].fillna(
    data['Location'].mode()[0] if not data['Location'].mode().empty else 'Unknown'
)

data['Transaction Date'] = pd.to_datetime(data['Transaction Date'], errors='coerce')
data['Transaction Date'] = data['Transaction Date'].fillna(
    data['Transaction Date'].median() if not data['Transaction Date'].isna().all() else pd.Timestamp('2023-01-01')
)

data['Discount Applied'] = data['Discount Applied'].fillna(False).astype(bool)


Removing 604 rows with 2 or more missing numerical values (Total Spent, Price Per Unit, Quantity):
      Transaction ID  Price Per Unit  Quantity  Total Spent
7        TXN_1372952            33.5       NaN          NaN
15       TXN_1809665            24.5       NaN          NaN
19       TXN_4206593            35.0       NaN          NaN
25       TXN_3481599            39.5       NaN          NaN
34       TXN_1621497            23.0       NaN          NaN
...              ...             ...       ...          ...
12527    TXN_1069238             5.0       NaN          NaN
12552    TXN_4823896             8.0       NaN          NaN
12556    TXN_4397672            41.0       NaN          NaN
12562    TXN_7422454            33.5       NaN          NaN
12564    TXN_2153066            29.0       NaN          NaN

[604 rows x 4 columns]


C:\Users\WINDOWS 11\AppData\Local\Temp\ipykernel_20448\500358002.py:69: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['Discount Applied'] = data['Discount Applied'].fillna(False).astype(bool)


In [130]:
data.isnull().sum()

Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64

In [131]:
# Validate Total Spent consistency
data['Calculated Total'] = data['Price Per Unit'] * data['Quantity']
inconsistent_rows = data[
    (data['Total Spent'].notna()) &
    (data['Calculated Total'].notna()) &
    (abs(data['Total Spent'] - data['Calculated Total']) > 0.01)
]

print("Inconsistent Total Spent Rows:")
print(inconsistent_rows[['Transaction ID', 'Price Per Unit', 'Quantity', 'Total Spent', 'Calculated Total']])

# Correct Total Spent
data.loc[
    (data['Total Spent'].notna()) &
    (data['Calculated Total'].notna()) &
    (abs(data['Total Spent'] - data['Calculated Total']) > 0.01),
    'Total Spent'
] = data['Calculated Total']

# Drop temporary column
data = data.drop(columns=['Calculated Total'])

Inconsistent Total Spent Rows:
Empty DataFrame
Columns: [Transaction ID, Price Per Unit, Quantity, Total Spent, Calculated Total]
Index: []


In [132]:
# Define function to detect outliers
def detect_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers

# Check outliers
for col in ['Price Per Unit', 'Quantity', 'Total Spent']:
    outliers = detect_outliers(data, col)
    print(f"Outliers in {col}: {len(outliers)} rows")
    print(outliers[[col, 'Transaction ID', 'Customer ID']].head())

# Cap outliers
for col in ['Price Per Unit', 'Quantity', 'Total Spent']:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data[col] = data[col].clip(lower=lower_bound, upper=upper_bound)

Outliers in Price Per Unit: 0 rows
Empty DataFrame
Columns: [Price Per Unit, Transaction ID, Customer ID]
Index: []
Outliers in Quantity: 0 rows
Empty DataFrame
Columns: [Quantity, Transaction ID, Customer ID]
Index: []
Outliers in Total Spent: 60 rows
      Total Spent Transaction ID Customer ID
27          410.0    TXN_1599706     CUST_14
133         410.0    TXN_2953434     CUST_25
339         410.0    TXN_4374445     CUST_12
869         410.0    TXN_1814138     CUST_06
1060        410.0    TXN_3710081     CUST_25


In [133]:
# Check for noise (negative or zero values)
noise = data[(data['Price Per Unit'] <= 0) | (data['Quantity'] <= 0) | (data['Total Spent'] <= 0)]
print("Noise (Negative/Zero Values):")
print(noise[['Transaction ID', 'Price Per Unit', 'Quantity', 'Total Spent']])

# Remove noisy rows
data = data[(data['Price Per Unit'] > 0) & (data['Quantity'] > 0) & (data['Total Spent'] > 0)]

Noise (Negative/Zero Values):
Empty DataFrame
Columns: [Transaction ID, Price Per Unit, Quantity, Total Spent]
Index: []


In [134]:
# Check duplicates
duplicates = data[data['Transaction ID'].duplicated()]
print("Duplicate Transaction IDs:")
print(duplicates[['Transaction ID', 'Customer ID']])

full_duplicates = data[data.duplicated()]
print("Fully Duplicate Rows:")
print(full_duplicates)

# Remove duplicates
data = data.drop_duplicates(subset=['Transaction ID'], keep='first')
data = data.drop_duplicates(keep='first')

Duplicate Transaction IDs:
Empty DataFrame
Columns: [Transaction ID, Customer ID]
Index: []
Fully Duplicate Rows:
Empty DataFrame
Columns: [Transaction ID, Customer ID, Category, Item, Price Per Unit, Quantity, Total Spent, Payment Method, Location, Transaction Date, Discount Applied]
Index: []


In [135]:
# Ensure Transaction Date is datetime
data['Transaction Date'] = pd.to_datetime(data['Transaction Date'], errors='coerce')

# Add Day of the Week
data['Day of the Week'] = data['Transaction Date'].dt.day_name()

# Add Transaction Month
data['Transaction Month'] = data['Transaction Date'].dt.month_name()

# Calculate Customer Lifetime Value (CLV) as total spent per customer
clv = data.groupby('Customer ID')['Total Spent'].sum().reset_index()
clv.columns = ['Customer ID', 'CLV']
data = data.merge(clv, on='Customer ID', how='left')

# Display the new columns
print("Dataset with New Columns:")
print(data[['Transaction ID', 'Day of the Week', 'Transaction Month', 'CLV']].head())

# Save to a new CSV file
output_file = 'retail_store_sales_enriched.csv'
data.to_csv(output_file, index=False)
print(f"\nDataset saved to: {output_file}")

Dataset with New Columns:


  Transaction ID Day of the Week Transaction Month      CLV
0    TXN_6867343          Monday             April  61417.0
1    TXN_3731986          Sunday              July  61719.5
2    TXN_9303719       Wednesday           October  62027.0
3    TXN_9458126        Saturday               May  58619.5
4    TXN_4575373          Sunday           October  66968.0

Dataset saved to: retail_store_sales_enriched.csv


In [136]:
data

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,Day of the Week,Transaction Month,CLV
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,Monday,April,61417.0
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,Sunday,July,61719.5
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,Wednesday,October,62027.0
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,False,Saturday,May,58619.5
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,Sunday,October,66968.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11966,TXN_9347481,CUST_18,Patisserie,Item_23_PAT,38.0,4.0,152.0,Credit Card,In-store,2023-09-03,False,Sunday,September,59135.5
11967,TXN_4009414,CUST_03,Beverages,Item_2_BEV,6.5,9.0,58.5,Cash,Online,2022-08-12,False,Friday,August,60791.5
11968,TXN_5306010,CUST_11,Butchers,Item_7_BUT,14.0,10.0,140.0,Cash,Online,2024-08-24,False,Saturday,August,60720.0
11969,TXN_5167298,CUST_04,Furniture,Item_7_FUR,14.0,6.0,84.0,Cash,Online,2023-12-30,True,Saturday,December,61748.0
